# MathVisionKid — chuẩn bị toàn bộ dữ liệu trên Colab

Notebook đọc toàn bộ member ứng viên được công bố theo STATE/PROGRESS và hash, gồm mọi gói bổ sung. Gói cân bằng nhỏ cũ giữ làm lịch sử/diagnostic, không dùng thay toàn kho.

Chỉ kiểm dữ liệu: không weights, trainer, GPU hoặc huấn luyện. Đăng nhập tài khoản có quyền đọc cùng ID Drive. Payload tạm nằm trong runtime Colab, tối đa một payload cache mỗi lúc, không tải kho về máy cá nhân.

Mỗi lần xử lý một đoạn tiếp nối; chạy tiếp sẽ tới các dòng/member chưa xử lý, không chọn tập nhỏ cố định. Train/validation/test giữ split gốc; sealed/pilot/challenge và history không tự vào train. Tổng candidate chưa phải tổng train-only đã kiểm. Full archive token/linkage/split audit vẫn pending.


In [ ]:
# Chỉ thư viện tokenizer đã pin từ pipeline.
%pip -q install transformers==5.16.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 88.0 MB/s eta 0:00:00


In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from pathlib import Path
import json,sys,sqlite3,hashlib
credentials, _ = google.auth.default()
drive_api = build('drive','v3',credentials=credentials,cache_discovery=False)
ROOT = Path('/content/mathvisionkid_data_only')
ROOT.mkdir(parents=True,exist_ok=True)
exec(compile('"""Account-neutral, data-only Colab bootstrap over fixed Drive IDs and hashes.\n\nNo training, weights, sharing edits or writes to Drive. Payloads are cached only\nin the caller\'s runtime directory; this module is not executed on the PC.\n"""\nfrom collections.abc import Mapping\nfrom pathlib import Path\nimport hashlib,io,json,zipfile\n\nSTATE_ID=\'1TLoQIbrRVmgsg-2uiUujCfM3HrK62Yme\'\nPROGRESS_ID=\'1D5RvQ5-V3DtYZfGxmlmvvPlVz9I-SxeX\'\nCODE_FILES=(\'exhaustive_colab_loader.py\',\'math_data_portable_resume.py\',\'math_data_archive_audit.py\',\'math_data_candidate_stream.py\')\ndef sha(data):return hashlib.sha256(data).hexdigest()\ndef verify(data,ref):\n    if len(data)!=ref[\'bytes\'] or sha(data)!=ref[\'sha256\']:raise ValueError(\'pinned bytes/SHA256 mismatch\')\n    return data\n\nclass GoogleDriveReader:\n    def __init__(self,service):self.service=service\n    def read(self,file_id,max_bytes):\n        from googleapiclient.http import MediaIoBaseDownload\n        meta=self.service.files().get(fileId=file_id,fields=\'id,size,mimeType\',supportsAllDrives=True).execute()\n        size=int(meta.get(\'size\',-1))\n        if size<0 or size>max_bytes:raise ValueError(\'Drive raw size exceeds bounded request\')\n        output=io.BytesIO();down=MediaIoBaseDownload(output,self.service.files().get_media(fileId=file_id,supportsAllDrives=True),chunksize=4*1024*1024)\n        done=False\n        while not done:\n            _,done=down.next_chunk(num_retries=3)\n            if output.tell()>max_bytes:raise ValueError(\'download exceeds bounded request\')\n        raw=output.getvalue()\n        if len(raw)!=size:raise ValueError(\'Drive raw size changed during download\')\n        return raw\n\ndef latest_controls(reader):\n    """Mutable STATE read twice; recipe itself is pinned by its exact hash."""\n    before=reader.read(STATE_ID,16*1024*1024);state=json.loads(before)\n    pointer=state[\'latest_all_data_union_recipe\']\n    if pointer[\'progress_id\']!=PROGRESS_ID:raise ValueError(\'unexpected progress control ID\')\n    member=pointer[\'member\']\n    if not isinstance(member,str) or member.startswith(\'/\') or \'..\' in Path(member).parts:raise ValueError(\'unsafe recipe member\')\n    raw=reader.read(PROGRESS_ID,128*1024*1024)\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        if archive.namelist().count(member)!=1:raise ValueError(\'missing/ambiguous latest union recipe\')\n        data=archive.read(member)\n    if sha(data)!=pointer[\'sha256\']:raise ValueError(\'STATE recipe hash differs from PROGRESS\')\n    recipe=json.loads(data)\n    if recipe.get(\'pending_discovery\'):raise ValueError(\'unresolved declared supplements; refusing partial data\')\n    if reader.read(STATE_ID,16*1024*1024)!=before:raise ValueError(\'STATE changed during read; resume from latest controls\')\n    if state.get(\'training_started\'):raise ValueError(\'unexpected training-started state for data-only notebook\')\n    return state,recipe,pointer\n\ndef bootstrap_code(reader,state,directory):\n    """Load exact code members from latest accepted bundle; no archive extractall."""\n    runs=[r for r in state.get(\'sgk_finish_runs\',[]) if r.get(\'bundle\')]\n    if not runs:raise ValueError(\'no explicit accepted code bundle\')\n    run=runs[-1];ref=run[\'bundle\'];fid=ref.get(\'id\',ref.get(\'file_id\'))\n    if not fid:raise ValueError(\'bundle file ID required\')\n    raw=verify(reader.read(fid,ref[\'bytes\']),ref)\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    receipts=[]\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        if len(archive.namelist())!=len(set(archive.namelist())):raise ValueError(\'duplicate bundle member\')\n        manifest=json.loads(archive.read(\'manifest.json\'))\n        for name in CODE_FILES:\n            path=\'code/\'+name;pin=manifest[\'members\'].get(path)\n            if not pin:raise ValueError(\'latest bundle lacks candidate reader dependency: \'+name)\n            data=verify(archive.read(path),pin)\n            (directory/name).write_bytes(data);receipts.append(dict(member=path,sha256=sha(data),bytes=len(data)))\n    return dict(bundle_id=fid,bundle_sha256=ref[\'sha256\'],code_members=receipts,training_started=False)\n\nclass LazyDriveCache(Mapping):\n    """One hash-verified payload cached at a time in ephemeral Colab storage.\n\n    source/chat members remain inside their archives. Checkpoints are separate\n    and never removed by cache eviction. No files are created on Drive.\n    """\n    def __init__(self,reader,recipe,directory):\n        self.reader=reader;self.directory=Path(directory);self.directory.mkdir(parents=True,exist_ok=True)\n        self.artifacts=recipe[\'artifacts\'];self.lookup={};self.current=None\n        for digest,ref in self.artifacts.items():\n            if digest!=ref[\'sha256\'] or not ref.get(\'file_ids\'):raise ValueError(\'hash and existing Drive ID required\')\n            self.lookup[digest]=digest\n            for fid in ref[\'file_ids\']:self.lookup[fid]=digest\n    def __iter__(self):return iter(self.lookup)\n    def __len__(self):return len(self.lookup)\n    def __getitem__(self,key):\n        digest=self.lookup[key];ref=self.artifacts[digest]\n        if self.current and self.current[0]==digest:\n            path=self.current[1]\n            if path.stat().st_size!=ref[\'bytes\'] or sha(path.read_bytes())!=digest:raise ValueError(\'cached payload changed\')\n            return path\n        data=verify(self.reader.read(ref[\'file_ids\'][0],ref[\'bytes\']),ref)\n        suffix=\'.gz\' if any(n.endswith(\'.gz\') for n in ref.get(\'file_names\',[])) else \'.zip\'\n        dest=self.directory/(digest+suffix);dest.write_bytes(data)\n        if self.current:\n            old=self.current[1]\n            if old.parent.resolve()!=self.directory.resolve():raise ValueError(\'cache deletion escaped runtime directory\')\n            old.unlink()\n        self.current=(digest,dest)\n        return dest\n\ndef scope_receipt(state,recipe,pointer):\n    return dict(accepted_cases=state[\'accepted_cases\'],declared_candidate_chat_rows=state[\'training_chat_rows\'],payload_references=len(recipe[\'artifacts\']),chat_members=len(recipe[\'chat_members\']),recipe_sha256=pointer[\'sha256\'],historical_train_only_total_verified=False,full_archive_token_split_audit_complete=False,TRAINING_DATASET_READY=False,training_started=False,selection=\'ALL_DECLARED_CANDIDATE_MEMBERS_NO_BALANCED_SUBSET\',runtime_accounts=\'Authenticate any account with access to the same existing IDs; no credentials in dataset\')\n', 'math_data_drive_bootstrap.py', 'exec'), globals())
reader = GoogleDriveReader(drive_api)
state,recipe,recipe_pointer = latest_controls(reader)
code_receipt = bootstrap_code(reader,state,ROOT/'code')
sys.path.insert(0,str(ROOT/'code'))
from math_data_candidate_stream import CandidateStream,DataOnlyConsumer
from exhaustive_colab_loader import load_qwen_tokenizer,tokenize_assistant_only
print(scope_receipt(state,recipe,recipe_pointer))


{'accepted_cases': 1629781, 'declared_candidate_chat_rows': 6501780, 'payload_references': 35, 'chat_members': 29, 'recipe_sha256': '7ef836cb65a8e5ca7c6bad5955d56d4f70b2d73fedccdb4e03a0038fe7b8e461', 'historical_train_only_total_verified': False, 'full_archive_token_split_audit_complete': False, 'TRAINING_DATASET_READY': False, 'training_started': False, 'selection': 'ALL_DECLARED_CANDIDATE_MEMBERS_NO_BALANCED_SUBSET', 'runtime_accounts': 'Authenticate any account with access to the same existing IDs; no credentials in dataset'}


Checkpoint chỉ chứa tiến độ/hash/ID. Mỗi recipe/split dùng một SQLite trong thư mục Drive gốc, chạy lại cập nhật cùng ID. Chỉ một notebook ghi mỗi lúc; SQLite không là khóa phân tán Drive. Recipe thay đổi cần chuyển checkpoint append-only, không xóa bản cũ để ép chạy.


In [ ]:
FOLDER_ID='19ZxkfKVIggdcbSfsAdCtG1ibfJ0CRN3n'
SPLIT='train'  # Chỉ đổi rõ thành validation/test khi cần kiểm split đó.
ROWS_PER_RUN=20000  # Đoạn công việc tiếp nối, không chọn tập train cố định.
TOKENIZE=True  # Chỉ tokenizer pinned, không model weights.
cache=LazyDriveCache(reader,recipe,ROOT/'payload_cache')
stream=CandidateStream(recipe,cache,split=SPLIT)
checkpoint_name=f'MathVisionKid_DATA_ONLY_{SPLIT}_{stream.identity[:24]}.sqlite'
query=f"'{FOLDER_ID}' in parents and name = '{checkpoint_name}' and trashed = false"
matches=drive_api.files().list(q=query,fields='files(id,name,size,appProperties)',supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get('files',[])
if len(matches)>1:
    raise RuntimeError('Checkpoint trùng tên: giữ nguyên tất cả, cần xác định ID trước khi tiếp.')
checkpoint_ref=matches[0] if matches else None
DB=ROOT/checkpoint_name
if not DB.exists() and checkpoint_ref:
    raw_db=reader.read(checkpoint_ref['id'],int(checkpoint_ref['size']))
    pin=checkpoint_ref.get('appProperties',{}).get('data_sha256')
    if not pin or hashlib.sha256(raw_db).hexdigest()!=pin:
        raise RuntimeError('Checkpoint thiếu hoặc sai hash; không tự khởi tạo lại.')
    DB.write_bytes(raw_db)
consumer=DataOnlyConsumer(DB,stream)
tokenizer=load_qwen_tokenizer() if TOKENIZE else None
span={'rows':0,'max_sequence_tokens':0,'max_completion_tokens':0,'source_links_not_reverified_here':0}
def data_only_check(row,provenance):
    if tokenizer is not None:
        item=tokenize_assistant_only(row,tokenizer)
        span['max_sequence_tokens']=max(span['max_sequence_tokens'],len(item['input_ids']))
        span['max_completion_tokens']=max(span['max_completion_tokens'],item['completion_tokens'])
    span['rows']+=1
    if provenance.get('sourcehash_origin')!='ORIGINAL_FIELD_VERIFIED_AGAINST_PINNED_SOURCE_MEMBER':
        span['source_links_not_reverified_here']+=1
try:
    progress=consumer.consume(data_only_check,max_rows=ROWS_PER_RUN)
finally:
    consumer.close()
print({'progress':progress,'this_work_span_only':span,'full_archive_linkage_split_audit_complete':False,'training_started':False})


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

{'progress': {'schema': 'MathVisionKid.candidate_consumer_progress.v1', 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train', 'counts': {'ACKNOWLEDGED_CANDIDATE': 20000}, 'acknowledged_rows': 20000, 'complete_members': 0, 'required_members': 29, 'incomplete_case_modes': 0, 'cursor': {'member_index': 0, 'next_line': 20000, 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train'}, 'stream_exhausted': False, 'all_data_training_ready': False, 'training_started': False}, 'this_work_span_only': {'rows': 20000, 'max_sequence_tokens': 1378, 'max_completion_tokens': 1213, 'source_links_not_reverified_here': 20000}, 'full_archive_linkage_split_audit_complete': False, 'training_started': False}


In [5]:
# Lưu cùng ID sau khi xử lý thành công và SQLite đã đóng.
from googleapiclient.http import MediaFileUpload
checkpoint_sha=hashlib.sha256(DB.read_bytes()).hexdigest()
properties={'data_sha256':checkpoint_sha,'recipe_identity':stream.identity,'split':SPLIT,'purpose':'DATA_ONLY_NO_TRAINING'}
if checkpoint_ref:
    fresh=drive_api.files().get(fileId=checkpoint_ref['id'],fields='id,appProperties',supportsAllDrives=True).execute()
    if fresh.get('appProperties',{}).get('data_sha256')!=checkpoint_ref.get('appProperties',{}).get('data_sha256'):
        raise RuntimeError('Checkpoint đã được cập nhật; không ghi đè tiến độ mới.')
    checkpoint_result=drive_api.files().update(fileId=checkpoint_ref['id'],body={'appProperties':properties},media_body=MediaFileUpload(str(DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
else:
    checkpoint_result=drive_api.files().create(body={'name':checkpoint_name,'parents':[FOLDER_ID],'appProperties':properties},media_body=MediaFileUpload(str(DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
verified_db=reader.read(checkpoint_result['id'],int(checkpoint_result['size']))
if hashlib.sha256(verified_db).hexdigest()!=checkpoint_sha:
    raise RuntimeError('Raw readback checkpoint không khớp; không tuyên bố lưu thành công.')
checkpoint_ref=checkpoint_result
receipt={'checkpoint_id':checkpoint_result['id'],'checkpoint_sha256':checkpoint_sha,'progress':progress,'this_work_span_only':span,'training_started':False,'TRAINING_DATASET_READY':False,'full_archive_token_linkage_split_audit_complete':False}
(ROOT/'last_data_only_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf-8')
print(receipt)


{'checkpoint_id': '1o3Q1uXA2ZTUAIHDpLCslHalg5qCbL-gr', 'checkpoint_sha256': '7b9babd8710bb598c2ec051073dc0d34de888686de97e3610cc10f1931b4a626', 'progress': {'schema': 'MathVisionKid.candidate_consumer_progress.v1', 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train', 'counts': {'ACKNOWLEDGED_CANDIDATE': 20000}, 'acknowledged_rows': 20000, 'complete_members': 0, 'required_members': 29, 'incomplete_case_modes': 0, 'cursor': {'member_index': 0, 'next_line': 20000, 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train'}, 'stream_exhausted': False, 'all_data_training_ready': False, 'training_started': False}, 'this_work_span_only': {'rows': 20000, 'max_sequence_tokens': 1378, 'max_completion_tokens': 1213, 'source_links_not_reverified_here': 20000}, 'training_started': False, 'TRAINING_DATASET_READY': False, 'full_archive_token_linkage_split_audit_complete': False}


Runtime mất kết nối: chạy lại để tải checkpoint đúng ID/hash và tiếp nối. stream_exhausted chỉ chứng minh đọc hết member của split/recipe này; không tự chứng minh mọi nguồn liên kết đúng hoặc các split độc lập. Cần source–chat/group/question/canonical audit theo queue trước công bố sẵn sàng train. Không đổi split hoặc bỏ bài hợp lệ để qua gate. Notebook chưa được chạy trên Colab trong phiên chuẩn bị này.
